# HMM allocation with long-short risk parity

A walk-forward HMM estimates regimes from volatility and semivariance. The allocation switches among hierarchical risk parity, long-short risk parity, and maximum Sharpe portfolios.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
REGIME-CONDITIONAL PORTFOLIO STRATEGIES WITH HAR-SEMIVARIANCE AND HMM
======================================================================

WALK-FORWARD VERSION - Look-ahead bias eliminated, HMM regime detection
WITH URTH BUY & HOLD BENCHMARK COMPARISON

MODIFICATIONS:
- Low Volatility (Regime 0): Max Sharpe on growth assets
- Normal Volatility (Regime 1): HRP on growth + defensive (NO SH)
- High Volatility (Regime 2): RISK PARITY with SHORT EXPOSURE via SH

NO LOOK-AHEAD BIAS - Walk-forward regime detection and feature engineering
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

# Import HMM library
try:
    from hmmlearn import hmm
except ImportError:
    print("ERROR: hmmlearn not installed. Install with: pip install hmmlearn")
    raise

# ============================================================================
# CONFIGURATION PARAMETERS (FULLY CONFIGURABLE)
# ============================================================================


# Date range for backtest
REGIME_TICKER = 'SPY'
BENCHMARK_TICKER = 'URTH'  # iShares MSCI World ETF for Buy & Hold comparison
START_DATE = '2013-01-01'
END_DATE = '2018-12-31'

# Asset universe - fully configurable
GROWTH_ASSETS = ['SPY', 'QQQ', 'IWM', 'VTI', 'XLK', 'XLV', 'XLF', 'XLE', 'AAPL', 'MSFT']
DEFENSIVE_ASSETS = ['TLT', 'GLD', 'UUP']  # REMOVED SH from general defensive assets
SHORT_ASSETS = ['SH']  # Inverse ETFs used ONLY in High Volatility regime
ALL_ASSETS = list(set(GROWTH_ASSETS + DEFENSIVE_ASSETS + SHORT_ASSETS))

# Feature engineering parameters
DAILY_LAG = 1
WEEKLY_LAG = 5
MONTHLY_LAG = 22
USE_VIX = True

# Regime detection parameters
N_REGIMES = 3
REGIME_REBALANCE_FREQ = 21  # Rebalance every 21 trading days
MIN_TRAIN_DAYS = 252  # Minimum training window for regime detection

# HMM-specific parameters - IMPROVED FOR CONVERGENCE
HMM_N_ITER = 200  # INCREASED: More iterations for better convergence
HMM_COVARIANCE_TYPE = 'diag'  # CHANGED: Diagonal covariance is more stable than 'full'
HMM_RANDOM_STATE = 42  # Fixed seed for reproducibility
HMM_TOL = 1e-3  # ADDED: Convergence tolerance
HMM_N_INIT = 5  # ADDED: Number of initialization attempts
HMM_MIN_COVAR = 1e-4  # ADDED: Minimum covariance to prevent numerical issues

# Portfolio construction parameters
MAX_HRP_ASSETS = 10
TRANSACTION_COST = 0.001  # 10 basis points per trade (realistic commission)
INITIAL_CAPITAL = 100000

# Risk Parity parameters for High Volatility regime
MIN_RISK_PARITY_WEIGHT = 0.05  # Minimum weight per asset (5%)
MAX_RISK_PARITY_WEIGHT = 0.40  # Maximum weight per asset (40%)

# ============================================================================
# DATA ACQUISITION
# ============================================================================

def fetch_price_data(ticker, start_date, end_date):
    try:
        frame = download_ohlcv(ticker.replace("%5E", "^"), start_date, end_date)
    except RuntimeError:
        return None
    return frame.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]


def fetch_all_assets(tickers, start_date, end_date):
    """
    Fetch all asset data from Yahoo Finance.

    Returns dictionary: {ticker: DataFrame}
    Only includes tickers with sufficient data (>200 days).
    NO BACKFILL OR FORWARD FILL - missing data is not imputed.
    """
    print(f"\n{'='*80}")
    print("FETCHING ASSET DATA")
    print(f"{'='*80}")

    asset_data = {}
    for ticker in tickers:
        try:
            print(f"  Fetching {ticker}...", end='')
            df = fetch_price_data(ticker, start_date, end_date)
            if df is not None and len(df) > 200:
                asset_data[ticker] = df
                print(f" ✓ {len(df)} days")
            else:
                print(f" ✗ Insufficient data")
        except Exception as e:
            print(f" ✗ Error: {str(e)}")

    print(f"\n✓ Successfully fetched {len(asset_data)} assets")
    return asset_data


# ============================================================================
# HAR-SEMIVARIANCE FEATURES (NO LOOK-AHEAD BIAS)
# ============================================================================

def calculate_garman_klass_vol(prices):
    """
    Calculate Garman-Klass volatility estimator.

    More efficient than close-to-close volatility as it uses OHLC data.
    Returns annualized volatility in percentage terms.

    Formula: sqrt(0.5 * (log(H/L))^2 - (2*log(2)-1) * (log(C/O))^2)

    This estimator is approximately 8x more efficient than standard volatility.
    """
    log_hl = np.log(prices['high'] / prices['low'])
    log_co = np.log(prices['close'] / prices['open'])
    gk_var = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

    # Annualize and convert to percentage
    rv_series = pd.Series(np.sqrt(gk_var.clip(lower=0) * 252) * 100, index=prices.index)
    return rv_series


def prepare_regime_features(spy_data, vix_data=None):
    """
    Prepare HAR-semivariance features for regime detection.

    ALL FEATURES ARE LAGGED TO AVOID LOOK-AHEAD BIAS:
    - Daily features: t-1 (yesterday's close)
    - Weekly features: rolling mean of t-1 to t-5
    - Monthly features: rolling mean of t-1 to t-22

    Features include:
    - RV (Realized Volatility) at D/W/M horizons
    - RS_neg (Downside Semivariance) at D/W horizons
    - VIX level at D/W horizons (if available)
    - Asymmetry ratio (downside/upside volatility)

    Returns DataFrame with features, indexed by date.
    NO BACKFILL OR FORWARD FILL is used.
    """

    print(f"\n{'='*80}")
    print("PREPARING REGIME DETECTION FEATURES")
    print(f"{'='*80}")

    df = pd.DataFrame(index=spy_data.index)

    # Calculate realized volatility from Garman-Klass estimator
    rv = calculate_garman_klass_vol(spy_data)
    returns = np.log(spy_data['close'] / spy_data['close'].shift(1))

    # Separate volatility into positive and negative return days (semivariances)
    # This captures asymmetric volatility patterns (downside risk vs upside risk)
    gk_var = (rv / (np.sqrt(252) * 100)) ** 2
    rs_pos_array = np.where(returns > 0, gk_var, 0)
    rs_neg_array = np.where(returns <= 0, gk_var, 0)

    rs_pos = pd.Series(np.sqrt(rs_pos_array * 252) * 100, index=spy_data.index)
    rs_neg = pd.Series(np.sqrt(rs_neg_array * 252) * 100, index=spy_data.index)

    # HAR components - ALL shifted by 1 day to avoid look-ahead bias
    # At time t, we only use information available up to t-1
    df['RV_D'] = rv.shift(DAILY_LAG)  # Yesterday's RV
    df['RV_W'] = rv.shift(1).rolling(WEEKLY_LAG).mean()  # Weekly average RV
    df['RV_M'] = rv.shift(1).rolling(MONTHLY_LAG).mean()  # Monthly average RV

    df['RS_neg_D'] = rs_neg.shift(DAILY_LAG)  # Yesterday's downside semivariance
    df['RS_neg_W'] = rs_neg.shift(1).rolling(WEEKLY_LAG).mean()  # Weekly average downside

    # VIX features (if available)
    if vix_data is not None:
        vix_aligned = vix_data.reindex(spy_data.index)
        df['VIX_level_D'] = vix_aligned.shift(DAILY_LAG)
        df['VIX_level_W'] = vix_aligned.shift(1).rolling(WEEKLY_LAG).mean()

    # Asymmetry ratio: downside vol / upside vol
    # High values indicate more downside risk (useful for regime detection)
    rs_pos_shifted = rs_pos.shift(DAILY_LAG)
    df['Asym_ratio_D'] = df['RS_neg_D'] / (rs_pos_shifted + 1e-8)

    # Drop rows with NaN values (from lagging and rolling windows)
    # NO BACKFILL OR FORWARD FILL - we simply remove incomplete observations
    df = df.dropna()

    print(f"✓ Prepared {len(df)} observations with {len(df.columns)} features")
    print(f"  Features: {list(df.columns)}")

    return df


# ============================================================================
# HMM REGIME DETECTION - IMPROVED CONVERGENCE
# ============================================================================

def initialize_hmm_params(X_train_scaled, n_regimes):
    """
    CRITICAL FIX: Initialize HMM parameters using K-means clustering.
    This provides a much better starting point than random initialization,
    significantly improving convergence stability and reproducibility.

    Parameters initialized:
    - startprob_: Initial state distribution (uniform)
    - transmat_: Transition matrix (persistence bias - regimes tend to persist)
    - means_: State means from K-means centroids
    - covars_: State covariances from K-means clusters

    Returns: dict with initialization parameters
    """
    from sklearn.cluster import KMeans

    # Use K-means to get initial estimates of regime centroids
    # This is much more stable than random initialization
    # n_init=10 ensures K-means itself converges to good solution
    kmeans = KMeans(n_clusters=n_regimes, random_state=HMM_RANDOM_STATE, n_init=10)
    kmeans_labels = kmeans.fit_predict(X_train_scaled)

    # Initialize means from K-means centroids
    means_init = kmeans.cluster_centers_

    # Initialize covariances from K-means clusters
    # Using diagonal covariance (variance of each feature independently)
    covars_init = np.zeros((n_regimes, X_train_scaled.shape[1]))
    for i in range(n_regimes):
        cluster_mask = kmeans_labels == i
        if cluster_mask.sum() > 1:
            # Use diagonal covariance (variance of each feature)
            cluster_data = X_train_scaled[cluster_mask]
            covars_init[i] = np.var(cluster_data, axis=0) + HMM_MIN_COVAR
        else:
            # If cluster has only 1 point, use overall variance
            covars_init[i] = np.var(X_train_scaled, axis=0) + HMM_MIN_COVAR

    # Initialize transition matrix with persistence bias
    # Regimes tend to persist (high diagonal values = 0.85)
    # This reflects the empirical observation that market regimes are sticky
    transmat_init = np.full((n_regimes, n_regimes), 0.05)
    np.fill_diagonal(transmat_init, 0.85)
    # Normalize rows to sum to 1 (required for stochastic matrix)
    transmat_init = transmat_init / transmat_init.sum(axis=1, keepdims=True)

    # Uniform initial state distribution (no prior bias)
    startprob_init = np.full(n_regimes, 1.0 / n_regimes)

    return {
        'startprob_': startprob_init,
        'transmat_': transmat_init,
        'means_': means_init,
        'covars_': covars_init
    }


def fit_hmm_with_multiple_inits(X_train_scaled, n_regimes, n_init_attempts=HMM_N_INIT):
    """
    CRITICAL FIX: Fit HMM with multiple initialization attempts.
    Returns the model with the best log-likelihood to avoid poor local optima.

    This addresses the core convergence issue by:
    1. Trying multiple initializations
    2. Using K-means-based initialization (not random)
    3. Selecting the best converged model based on log-likelihood
    4. Handling convergence failures gracefully

    Returns: best_model, best_score, converged
    """
    best_model = None
    best_score = -np.inf
    best_converged = False

    for attempt in range(n_init_attempts):
        try:
            # Create model with improved parameters
            model = hmm.GaussianHMM(
                n_components=n_regimes,
                covariance_type=HMM_COVARIANCE_TYPE,  # 'diag' for numerical stability
                n_iter=HMM_N_ITER,
                tol=HMM_TOL,  # ADDED: Explicit convergence tolerance
                random_state=HMM_RANDOM_STATE + attempt,  # Different seed per attempt
                verbose=False,
                init_params='',  # CRITICAL: We initialize manually below
                params='stmc'  # Estimate all parameters: startprob, transmat, means, covars
            )

            # Initialize parameters using K-means (MUCH better than random init)
            init_params = initialize_hmm_params(X_train_scaled, n_regimes)
            model.startprob_ = init_params['startprob_']
            model.transmat_ = init_params['transmat_']
            model.means_ = init_params['means_']
            model.covars_ = init_params['covars_']

            # Fit the model
            model.fit(X_train_scaled)

            # Check if model converged by computing log-likelihood
            score = model.score(X_train_scaled)

            # Update best model if this one is better
            if score > best_score and model.monitor_.converged:
                best_score = score
                best_model = model
                best_converged = True
            elif score > best_score and not best_converged:
                # Even if not fully converged, keep best attempt
                best_score = score
                best_model = model

        except Exception as e:
            # If this initialization fails, try next one
            continue

    return best_model, best_score, best_converged


def detect_regime_walk_forward_hmm(features, current_date, n_regimes=3, min_train_days=252):
    """
    Detect regime at current_date using ONLY data up to (but not including) current_date.

    MAJOR IMPROVEMENTS FOR CONVERGENCE:
    1. Multiple initialization attempts with K-means seeding
    2. Explicit tolerance and iteration limits
    3. Diagonal covariance for numerical stability
    4. Proper parameter initialization
    5. Best model selection based on log-likelihood

    Walk-forward approach with HMM:
    1. Select training data: all dates < current_date
    2. Fit StandardScaler on training data
    3. Fit GaussianHMM on scaled training data (with multiple attempts)
    4. Map HMM states to regimes (0=low vol, 1=normal, 2=high vol)
    5. Predict regime for current_date

    HMM provides better temporal modeling compared to K-Means by:
    - Modeling transition probabilities between regimes
    - Capturing time-series dependencies
    - Providing probabilistic state assignments

    Returns:
    - regime_label: int (0, 1, or 2) representing low/normal/high volatility
    - scaler: fitted StandardScaler (for reference)
    - model: fitted HMM model (for reference)

    Returns (None, None, None) if insufficient training data or convergence failure.
    """
    # Get all dates STRICTLY before current_date (no look-ahead bias)
    train_mask = features.index < current_date
    train_data = features.loc[train_mask]

    # Need sufficient training data for stable HMM estimation
    if len(train_data) < min_train_days:
        return None, None, None

    # Fit scaler on training data only
    X_train = train_data.values
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    try:
        # FIT HMM WITH MULTIPLE ATTEMPTS (CRITICAL FIX FOR CONVERGENCE)
        model, score, converged = fit_hmm_with_multiple_inits(
            X_train_scaled, n_regimes, n_init_attempts=HMM_N_INIT
        )

        if model is None:
            return None, None, None

        # Predict hidden states for training data
        train_states = model.predict(X_train_scaled)

        # Map HMM states to regimes by sorting on mean RV_D in training period
        # This ensures regime 0 = low vol, regime 1 = normal vol, regime 2 = high vol
        regime_means = []
        for i in range(n_regimes):
            state_mask = train_states == i
            if state_mask.sum() > 0:
                mean_rv = train_data.loc[train_data.index[state_mask], 'RV_D'].mean()
                regime_means.append((i, mean_rv))

        # Sort by mean volatility to create ordered regimes
        regime_means_sorted = sorted(regime_means, key=lambda x: x[1])
        state_to_regime = {old: new for new, (old, _) in enumerate(regime_means_sorted)}

        # Now predict regime for current_date
        if current_date not in features.index:
            return None, scaler, model

        # Get features for current date and predict
        X_current = features.loc[current_date:current_date].values
        X_current_scaled = scaler.transform(X_current)

        # Predict hidden state for current date
        state_label = model.predict(X_current_scaled)[0]
        regime_label = state_to_regime[state_label]

        return regime_label, scaler, model

    except Exception as e:
        print(f"  HMM fitting error: {str(e)}")
        return None, None, None


# ============================================================================
# CANONICAL HRP IMPLEMENTATION
# ============================================================================

def get_quasi_diag(link):
    """
    Reorder assets based on hierarchical tree clustering.

    This implements the quasi-diagonalization step of HRP.
    Returns list of asset indices in quasi-diagonal order.

    The quasi-diagonal ordering places similar assets close together,
    which is essential for the recursive bisection step.
    """
    link = link.astype(int)
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    num_items = link[-1, 3]

    while sort_ix.max() >= num_items:
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        df0 = sort_ix[sort_ix >= num_items]
        i = df0.index
        j = df0.values - num_items
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sort_ix = pd.concat([sort_ix, df0])
        sort_ix = sort_ix.sort_index()
        sort_ix.index = range(sort_ix.shape[0])

    return sort_ix.tolist()


def get_cluster_var(cov, c_items):
    """
    Compute variance of a cluster using inverse-variance weights.

    This is used in the recursive bisection step to determine
    how to allocate weight between two clusters.

    Lower cluster variance -> higher weight allocation.
    """
    cov_slice = cov.loc[c_items, c_items]
    inv_diag = 1 / np.diag(cov_slice)
    w = inv_diag / inv_diag.sum()
    c_var = np.dot(w, np.dot(cov_slice, w))
    return c_var


def get_rec_bipart(cov, sort_ix):
    """
    Recursive bisection to compute HRP weights.

    This is the CANONICAL HRP algorithm from Lopez de Prado:
    1. Start with all assets, each with weight = 1
    2. Recursively bisect clusters into two sub-clusters
    3. Allocate weight between sub-clusters based on inverse cluster variance
    4. Continue until each cluster contains exactly one asset

    The result is a diversified portfolio that respects the hierarchical
    structure of asset correlations.
    """
    w = pd.Series(1, index=sort_ix)
    c_items = [sort_ix]

    # Recursive bisection loop
    while len(c_items) > 0:
        # Split each cluster into two halves
        c_items = [i[j:k] for i in c_items 
                   for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) 
                   if len(i) > 1]

        # Allocate weights between each pair of sub-clusters
        for i in range(0, len(c_items), 2):
            c_items0 = c_items[i]      # Left sub-cluster
            c_items1 = c_items[i + 1]  # Right sub-cluster

            # Compute cluster variances
            c_var0 = get_cluster_var(cov, c_items0)
            c_var1 = get_cluster_var(cov, c_items1)

            # Allocate weight: inverse variance weighting
            alpha = 1 - c_var0 / (c_var0 + c_var1)

            # Scale weights for both sub-clusters
            w[c_items0] *= alpha       # Left cluster gets alpha
            w[c_items1] *= 1 - alpha   # Right cluster gets 1-alpha

    return w


def hrp_portfolio_canonical(returns, max_assets=None):
    """
    Canonical Hierarchical Risk Parity portfolio implementation.

    Three-step process:
    1. Tree Clustering: Build hierarchical tree using correlation distance
    2. Quasi-Diagonalization: Reorder assets based on tree structure
    3. Recursive Bisection: Allocate weights using inverse cluster variance

    This implementation follows Lopez de Prado's original HRP algorithm.

    Used in NORMAL VOLATILITY regime (balanced diversification).

    Parameters:
    - returns: DataFrame of asset returns
    - max_assets: Optional limit on number of assets (selects top N by weight)

    Returns dict: {asset: weight}
    """
    # Need sufficient data for stable covariance estimation
    if len(returns) < 60:
        return {}

    try:
        # Step 1: Hierarchical Tree Clustering
        # Use correlation distance: d = sqrt((1 - corr) / 2)
        corr = returns.corr()
        dist = np.sqrt((1 - corr) / 2)

        # Convert to condensed distance matrix for scipy
        dist_condensed = squareform(dist.values, checks=False)

        # Single linkage clustering (as per original HRP)
        link = linkage(dist_condensed, method='single')

        # Step 2: Quasi-Diagonalization
        sort_ix = get_quasi_diag(link)
        sort_ix = [returns.columns[i] for i in sort_ix]

        # Step 3: Recursive Bisection
        # Use annualized covariance matrix
        cov = returns.cov() * 252
        weights = get_rec_bipart(cov, sort_ix)

        # Optional: Limit to top N assets by weight to reduce turnover
        if max_assets is not None and len(weights) > max_assets:
            weights = weights.nlargest(max_assets)
            weights = weights / weights.sum()

        # Filter out negligible weights (< 1%)
        return {asset: w for asset, w in weights.items() if w > 0.01}

    except Exception as e:
        print(f"  HRP error: {str(e)}")
        return {}


# ============================================================================
# RISK PARITY PORTFOLIO WITH SHORT EXPOSURE (NEW FOR HIGH VOL REGIME)
# ============================================================================

def risk_parity_with_short_portfolio(returns, defensive_assets, short_assets, 
                                     min_weight=MIN_RISK_PARITY_WEIGHT, 
                                     max_weight=MAX_RISK_PARITY_WEIGHT):
    """
    Risk Parity Portfolio with SHORT EXPOSURE via inverse ETF (SH).

    STRATEGY FOR HIGH VOLATILITY REGIME:
    ------------------------------------
    Objective: Achieve equal risk contribution across defensive AND short assets.

    This strategy is appropriate during high volatility because:
    1. Risk Parity balances risk across all positions (including shorts)
    2. SH provides negative correlation to equities (downside protection)
    3. Equal risk contribution ensures no single asset dominates portfolio risk
    4. Combines defensive assets (TLT, GLD, UUP) with short equity exposure (SH)

    Mathematical Formulation:
    -------------------------
    For each asset i, the risk contribution is:
        RC_i = w_i * (Cov * w)_i / sqrt(w' * Cov * w)

    Goal: RC_1 = RC_2 = ... = RC_N (equal risk contribution)

    This is achieved by minimizing:
        sum((RC_i - target_RC)^2)

    Subject to:
        - sum(w_i) = 1 (fully invested)
        - min_weight <= w_i <= max_weight (position limits)

    Parameters:
    -----------
    returns : pd.DataFrame
        Historical returns of all assets
    defensive_assets : list
        List of defensive asset tickers (TLT, GLD, UUP)
    short_assets : list
        List of inverse ETF tickers (SH)
    min_weight : float
        Minimum weight per asset (default 5%)
    max_weight : float
        Maximum weight per asset (default 40%)

    Returns:
    --------
    dict : {ticker: weight}
        Portfolio weights achieving equal risk contribution
    """
    # Filter to high volatility universe: defensive + short
    high_vol_universe = defensive_assets + short_assets
    available_assets = [a for a in high_vol_universe if a in returns.columns]

    if len(available_assets) < 2:
        # Not enough assets - fallback to equal weight
        return {asset: 1.0 / len(available_assets) for asset in available_assets}

    returns_subset = returns[available_assets]

    # Need sufficient data for stable covariance estimation
    if len(returns_subset) < 60:
        return {asset: 1.0 / len(available_assets) for asset in available_assets}

    try:
        # Calculate annualized covariance matrix
        # Using Ledoit-Wolf shrinkage for robust estimation
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns_subset.values).covariance_ * 252
        cov_matrix = pd.DataFrame(cov_matrix, index=available_assets, columns=available_assets)

        n_assets = len(available_assets)

        # Objective function: minimize sum of squared deviations from equal risk contribution
        def risk_parity_objective(weights):
            # Portfolio volatility
            port_var = np.dot(weights, np.dot(cov_matrix, weights))
            port_vol = np.sqrt(port_var)

            if port_vol < 1e-8:
                return 1e10  # Penalize degenerate solutions

            # Marginal contribution to risk for each asset
            marginal_contrib = np.dot(cov_matrix, weights)

            # Risk contribution for each asset: RC_i = w_i * MCR_i / portfolio_vol
            risk_contrib = weights * marginal_contrib / port_vol

            # Target: equal risk contribution
            target_rc = port_vol / n_assets

            # Minimize sum of squared differences from target
            return np.sum((risk_contrib - target_rc) ** 2)

        # Constraints: weights sum to 1
        constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0}]

        # Bounds: min_weight <= w_i <= max_weight
        bounds = [(min_weight, max_weight) for _ in range(n_assets)]

        # Initial guess: equal weight (naive risk parity starting point)
        x0 = np.array([1.0 / n_assets] * n_assets)

        # Optimize using SLSQP (Sequential Least Squares Programming)
        result = minimize(
            risk_parity_objective,
            x0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )

        if not result.success:
            # Optimization failed - fallback to equal weight
            print(f"  Risk Parity optimization failed: {result.message}")
            return {asset: 1.0 / n_assets for asset in available_assets}

        weights = result.x

        # Filter out negligible weights and renormalize
        weights_dict = {asset: w for asset, w in zip(available_assets, weights) if w > 0.01}
        total_weight = sum(weights_dict.values())
        if total_weight > 0:
            weights_dict = {k: v / total_weight for k, v in weights_dict.items()}

        return weights_dict

    except Exception as e:
        print(f"  Risk Parity error: {str(e)}")
        # Fallback to equal weight
        return {asset: 1.0 / len(available_assets) for asset in available_assets}


# ============================================================================
# MAX SHARPE PORTFOLIO (LOW VOLATILITY REGIME)
# ============================================================================

def max_sharpe_portfolio(returns, risk_free_rate=0.02):
    """
    Maximum Sharpe Ratio portfolio optimization.

    Used in LOW VOLATILITY regime to maximize risk-adjusted returns.

    Optimization problem:
    - Maximize: (portfolio_return - risk_free_rate) / portfolio_volatility
    - Subject to: sum(weights) = 1, weights >= 0, individual weight <= 30%

    Uses Ledoit-Wolf shrinkage for robust covariance estimation.

    Parameters:
    - returns: DataFrame of asset returns
    - risk_free_rate: Annual risk-free rate (default 2%)

    Returns dict: {asset: weight}
    """
    # Need sufficient data for stable optimization
    if len(returns) < 60:
        return {}

    # Compute expected returns (annualized mean)
    mu = returns.mean() * 252

    try:
        # Use Ledoit-Wolf shrinkage for robust covariance estimation
        # This reduces estimation error in covariance matrix
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns.values).covariance_ * 252
    except:
        return {}

    n_assets = len(mu)

    # Objective: negative Sharpe ratio (we minimize negative = maximize positive)
    def neg_sharpe(weights):
        port_return = np.dot(weights, mu)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(port_return - risk_free_rate) / port_vol if port_vol > 0 else -np.inf

    # Constraints: weights sum to 1
    constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0}]

    # Bounds: long only, max 30% per asset to ensure diversification
    bounds = [(0, 0.3) for _ in range(n_assets)]

    # Initial guess: equal weight
    x0 = np.array([1.0 / n_assets] * n_assets)

    # Run optimization using SLSQP
    result = minimize(neg_sharpe, x0, method='SLSQP', bounds=bounds, 
                     constraints=constraints, options={'maxiter': 1000})

    if result.success:
        weights = result.x
        # Filter out negligible weights (< 1%)
        weights = np.where(weights < 0.01, 0, weights)
        if np.sum(weights) > 0:
            weights = weights / np.sum(weights)
            return {asset: w for asset, w in zip(returns.columns, weights) if w > 0.01}

    return {}


# ============================================================================
# BENCHMARK: BUY & HOLD STRATEGY
# ============================================================================

def create_buy_hold_benchmark(benchmark_data, regime_dates, initial_capital, transaction_cost):
    """
    Create Buy & Hold benchmark strategy for comparison.

    Strategy:
    - Buy at first available date with realistic transaction cost
    - Hold until end (no rebalancing)
    - Account for transaction costs on initial purchase only

    This represents a passive investment approach for comparison with
    the active regime-conditional strategy.

    Parameters:
    - benchmark_data: DataFrame with OHLC data for benchmark ticker
    - regime_dates: List of dates to align with strategy
    - initial_capital: Starting capital
    - transaction_cost: Transaction cost rate (e.g., 0.001 = 0.1%)

    Returns DataFrame with columns: date, value, return
    """
    print(f"\n{'='*80}")
    print("CREATING BUY & HOLD BENCHMARK")
    print(f"{'='*80}")

    # Align benchmark data to regime dates
    benchmark_aligned = benchmark_data.reindex(regime_dates)

    # Find first valid date with price data
    first_valid_idx = benchmark_aligned['close'].first_valid_index()
    if first_valid_idx is None:
        print("  ✗ No valid benchmark data")
        return None

    first_price = benchmark_aligned.loc[first_valid_idx, 'close']

    # Account for transaction cost on initial purchase (realistic commission)
    net_capital = initial_capital * (1 - transaction_cost)
    shares = net_capital / first_price

    print(f"  Initial purchase: {shares:.2f} shares at ${first_price:.2f}")
    print(f"  Transaction cost: ${initial_capital * transaction_cost:.2f}")

    # Calculate portfolio value for each date
    benchmark_values = []
    for date in regime_dates:
        if date >= first_valid_idx and not pd.isna(benchmark_aligned.loc[date, 'close']):
            current_price = benchmark_aligned.loc[date, 'close']
            portfolio_value = shares * current_price

            benchmark_values.append({
                'date': date,
                'value': portfolio_value,
                'price': current_price
            })

    benchmark_df = pd.DataFrame(benchmark_values)
    benchmark_df['return'] = benchmark_df['value'].pct_change()

    print(f"  ✓ Benchmark created: {len(benchmark_df)} days")
    print(f"  Final value: ${benchmark_df['value'].iloc[-1]:,.2f}")

    return benchmark_df


# ============================================================================
# BACKTEST ENGINE WITH WALK-FORWARD HMM REGIME DETECTION
# ============================================================================

class RegimePortfolioBacktest_WalkForwardHMM:
    """
    Backtest engine with walk-forward HMM regime detection.

    NO LOOK-AHEAD BIAS IMPLEMENTATION:
    -----------------------------------
    - Regime detection uses only data up to t-1
    - Portfolio optimization uses only data up to t-1
    - Execution at t uses open price (known at market open)
    - Portfolio valuation at t uses close price

    Trading Flow:
    -------------
    Day T-1 (Close):
    - Portfolio valued at Close[T-1]
    - Features computed from T-2 data
    - Regime detected using data up to T-1
    - Optimization using returns up to T-1
    - Signals generated for T

    Day T (Open):
    - Execute pending signals at Open[T]
    - Use Close[T-1] for position sizing (no look-ahead)
    - Portfolio positions updated
    - Realistic transaction costs applied

    Day T (Close):
    - Portfolio valued at Close[T]
    - Returns: Close[T]/Close[T-1] - 1
    - This value becomes Close[T-1] for T+1 sizing
    """

    def __init__(self, regime_features, asset_data, initial_capital, transaction_cost):
        self.regime_features = regime_features
        self.asset_data = asset_data
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost

        self.capital = initial_capital
        self.cash = initial_capital
        self.positions = {}  # {ticker: shares}
        self.portfolio_history = []
        self.trades = []
        self.regime_history = []

        self.align_asset_data()

    def align_asset_data(self):
        """Align all asset data to common regime feature dates."""
        common_dates = self.regime_features.index
        for ticker in self.asset_data:
            self.asset_data[ticker] = self.asset_data[ticker].reindex(common_dates)

    def get_current_prices(self, date, price_type='close'):
        """
        Get current prices for all assets at given date.

        Parameters:
        - date: Date to get prices for
        - price_type: 'open' or 'close'

        Returns dict: {ticker: price}
        """
        prices = {}
        for ticker, df in self.asset_data.items():
            if date in df.index and not pd.isna(df.loc[date, price_type]):
                prices[ticker] = df.loc[date, price_type]
        return prices

    def calculate_portfolio_value(self, date):
        """
        Calculate total portfolio value at given date.

        Portfolio value = cash + sum(shares * close_price)
        Uses close prices for valuation.
        """
        prices = self.get_current_prices(date, 'close')
        risky_value = sum(self.positions.get(ticker, 0) * prices.get(ticker, 0) 
                         for ticker in self.positions)
        return self.cash + risky_value

    def execute_rebalance(self, date, target_weights):
        """
        Execute portfolio rebalancing with realistic timing and transaction costs.

        IMPORTANT: To avoid look-ahead bias:
        - Decision made at date (using close price for valuation)
        - Position sizing based on Close[T-1] value (no look-ahead)
        - Execution at next date's open price (realistic market execution)
        - Realistic transaction costs applied to ALL trades

        Parameters:
        - date: Current date (decision date at close)
        - target_weights: Dict {ticker: weight} representing target allocation
        """
        dates_list = list(self.regime_features.index)
        current_idx = dates_list.index(date)

        # Need next day for execution
        if current_idx >= len(dates_list) - 1:
            return

        next_date = dates_list[current_idx + 1]

        # Execute at next day's OPEN price (realistic execution timing)
        exec_prices = self.get_current_prices(next_date, 'open')

        if not exec_prices:
            return

        # Calculate portfolio value at current date's CLOSE
        # This is the capital available for rebalancing (NO LOOK-AHEAD)
        portfolio_value = self.calculate_portfolio_value(date)

        # Calculate target positions in shares
        target_positions = {}
        for ticker, weight in target_weights.items():
            if ticker in exec_prices:
                target_value = portfolio_value * weight
                target_positions[ticker] = target_value / exec_prices[ticker]

        # Step 1: Close positions not in target portfolio (sell unwanted holdings)
        for ticker in list(self.positions.keys()):
            if ticker not in target_positions and self.positions[ticker] != 0:
                if ticker in exec_prices:
                    sell_value = self.positions[ticker] * exec_prices[ticker]
                    cost = abs(sell_value) * self.transaction_cost  # Realistic commission
                    self.cash += sell_value - cost
                    self.positions[ticker] = 0

                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': sell_value,
                        'cost': cost
                    })

        # Step 2: Adjust to target positions (buy/sell to reach targets)
        for ticker, target_shares in target_positions.items():
            current_shares = self.positions.get(ticker, 0)
            delta_shares = target_shares - current_shares

            # Only trade if delta is significant (avoid tiny trades that incur costs)
            if abs(delta_shares) > 0.01:
                trade_value = delta_shares * exec_prices[ticker]
                cost = abs(trade_value) * self.transaction_cost  # Realistic commission

                if trade_value > 0:  # BUY
                    # Check if we have enough cash (cash constraint)
                    if self.cash >= (trade_value + cost):
                        self.positions[ticker] = target_shares
                        self.cash -= (trade_value + cost)

                        self.trades.append({
                            'date': next_date,
                            'ticker': ticker,
                            'action': 'BUY',
                            'value': trade_value,
                            'cost': cost
                        })

                else:  # SELL
                    self.positions[ticker] = target_shares
                    self.cash += (-trade_value - cost)

                    self.trades.append({
                        'date': next_date,
                        'ticker': ticker,
                        'action': 'SELL',
                        'value': trade_value,
                        'cost': cost
                    })

    def run_backtest(self):
        """
        Execute full backtest with walk-forward HMM regime detection.

        Process:
        --------
        1. For each date, detect regime using only past data (walk-forward)
        2. Every REGIME_REBALANCE_FREQ days, compute optimal portfolio for current regime
        3. Execute rebalancing at next day's open (realistic timing)
        4. Apply realistic transaction costs to all trades
        5. Track portfolio value, trades, and regime history

        Returns:
        --------
        - portfolio_history: List of daily portfolio snapshots
        - trades: List of all trades executed (with costs)
        """

        print(f"\n{'='*80}")
        print("RUNNING WALK-FORWARD HMM REGIME-CONDITIONAL BACKTEST")
        print(f"{'='*80}")

        dates = list(self.regime_features.index)
        last_rebalance = -REGIME_REBALANCE_FREQ  # Force rebalance on first opportunity

        for day_num, date in enumerate(dates):
            # ================================================================
            # REGIME DETECTION: Walk-forward HMM using only data up to current date
            # This ensures no look-ahead bias in regime classification
            # ================================================================
            regime_label, scaler, hmm_model = detect_regime_walk_forward_hmm(
                self.regime_features, date, N_REGIMES, MIN_TRAIN_DAYS
            )

            if regime_label is None:
                # Not enough training data yet - stay in cash
                portfolio_value = self.calculate_portfolio_value(date)
                self.portfolio_history.append({
                    'date': date,
                    'value': portfolio_value,
                    'cash': self.cash,
                    'regime': None,
                    'regime_name': 'Insufficient Data',
                    'num_positions': len([p for p in self.positions.values() if p != 0])
                })
                continue

            # Map regime label to descriptive name
            regime_name = ['Low Volatility', 'Normal Volatility', 'High Volatility'][regime_label]

            # Store regime for analysis
            self.regime_history.append({
                'date': date,
                'regime': regime_label,
                'regime_name': regime_name
            })

            # Calculate current portfolio value
            portfolio_value = self.calculate_portfolio_value(date)

            # ================================================================
            # REBALANCING LOGIC: Check if it's time to rebalance
            # ================================================================
            days_since_rebal = day_num - last_rebalance

            if days_since_rebal >= REGIME_REBALANCE_FREQ and day_num >= MIN_TRAIN_DAYS + 60:
                # ============================================================
                # PORTFOLIO OPTIMIZATION: Use historical returns up to current date
                # Lookback window: up to 252 days (1 year) for stable estimates
                # ============================================================
                hist_start = max(0, day_num - 252)
                hist_dates = dates[hist_start:day_num]

                # Compute returns for all assets using ONLY historical data (NO LOOK-AHEAD)
                returns_list = []
                for ticker in self.asset_data:
                    ticker_prices = self.asset_data[ticker].loc[hist_dates, 'close']
                    ticker_returns = ticker_prices.pct_change().dropna()
                    returns_list.append(ticker_returns.rename(ticker))

                if returns_list:
                    returns_df = pd.concat(returns_list, axis=1).dropna()

                    # ========================================================
                    # REGIME-CONDITIONAL PORTFOLIO ALLOCATION (MODIFIED)
                    # ========================================================

                    if regime_label == 0:
                        # LOW VOLATILITY: Maximum Sharpe Ratio on growth assets
                        # Objective: Maximize risk-adjusted returns
                        # Rationale: In low volatility, maximize exposure to growth
                        growth_returns = returns_df[[c for c in GROWTH_ASSETS if c in returns_df.columns]]
                        target_weights = max_sharpe_portfolio(growth_returns)
                        strategy_name = "Max Sharpe"

                    elif regime_label == 1:
                        # NORMAL VOLATILITY: Canonical HRP on growth + defensive (NO SH)
                        # Objective: Balanced diversification using hierarchical structure
                        # Rationale: Balanced exposure across asset classes
                        # IMPORTANT: SH is EXCLUDED from Normal Vol to avoid bearish drag
                        normal_vol_assets = GROWTH_ASSETS + DEFENSIVE_ASSETS
                        all_returns = returns_df[[c for c in normal_vol_assets if c in returns_df.columns]]
                        target_weights = hrp_portfolio_canonical(all_returns, max_assets=MAX_HRP_ASSETS)
                        strategy_name = "HRP"

                    else:  # regime_label == 2
                        # HIGH VOLATILITY: RISK PARITY with SHORT EXPOSURE via SH
                        # Objective: Capital preservation with downside protection
                        # Rationale: Equal risk contribution + short equity exposure for crisis periods
                        # MODIFICATION: Uses Risk Parity instead of MST
                        target_weights = risk_parity_with_short_portfolio(
                            returns_df,
                            defensive_assets=DEFENSIVE_ASSETS,
                            short_assets=SHORT_ASSETS,
                            min_weight=MIN_RISK_PARITY_WEIGHT,
                            max_weight=MAX_RISK_PARITY_WEIGHT
                        )
                        strategy_name = "Risk Parity + Short"

                    # Execute rebalancing if we have valid weights
                    if target_weights:
                        print(f"\n  Day {day_num}: {date.strftime('%Y-%m-%d')} - {regime_name}")
                        print(f"    Strategy: {strategy_name}")
                        print(f"    Assets: {list(target_weights.keys())}")
                        print(f"    Weights: {{{', '.join([f'{k}: {v:.1%}' for k, v in target_weights.items()])}}}")

                        self.execute_rebalance(date, target_weights)
                        last_rebalance = day_num

            # Record portfolio state
            self.portfolio_history.append({
                'date': date,
                'value': portfolio_value,
                'cash': self.cash,
                'regime': regime_label,
                'regime_name': regime_name,
                'num_positions': len([p for p in self.positions.values() if p != 0])
            })

            # Progress update every 100 days
            if day_num % 100 == 0:
                print(f"  Progress: {day_num}/{len(dates)} days ({(day_num/len(dates)*100):.1f}%)")

        print(f"\n✓ Backtest complete (Walk-Forward HMM Regime Detection)")
        return self.portfolio_history, self.trades


# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

def calculate_performance_metrics(portfolio_df, initial_capital):
    """
    Calculate comprehensive performance metrics.

    Metrics:
    --------
    - Total Return (%): (Final Value - Initial Capital) / Initial Capital
    - Annualized Return (%): Mean daily return * 252 trading days
    - Annualized Volatility (%): Std dev of daily returns * sqrt(252)
    - Sharpe Ratio: (Annual Return - 2%) / Annual Volatility
    - Maximum Drawdown (%): Largest peak-to-trough decline
    - Calmar Ratio: Annual Return / |Maximum Drawdown|
    - Win Rate (%): Percentage of days with positive returns

    Returns dict of metrics.
    """
    final_value = portfolio_df['value'].iloc[-1]
    total_return = (final_value / initial_capital - 1) * 100

    returns = portfolio_df['value'].pct_change().dropna()

    # Annualized metrics (assuming 252 trading days per year)
    annual_return = returns.mean() * 252 * 100
    annual_vol = returns.std() * np.sqrt(252) * 100
    sharpe = (annual_return - 2.0) / annual_vol if annual_vol > 0 else 0

    # Drawdown analysis
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min() * 100

    # Calmar ratio: return / max drawdown (risk-adjusted performance)
    calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

    # Win rate
    win_rate = (returns > 0).sum() / len(returns) * 100 if len(returns) > 0 else 0

    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_vol': annual_vol,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'calmar': calmar,
        'win_rate': win_rate,
        'final_value': final_value
    }


def print_comparative_metrics(strategy_metrics, benchmark_metrics):
    """
    Print side-by-side comparison of strategy vs benchmark metrics.

    Formats metrics in a readable table for easy comparison.
    Shows absolute differences for dollar values and percentage point differences for ratios.
    """
    print(f"\n{'='*80}")
    print("COMPARATIVE PERFORMANCE: STRATEGY VS BENCHMARK")
    print(f"{'='*80}")
    print(f"\n{'Metric':<25} {'Strategy':>15} {'Benchmark':>15} {'Difference':>15}")
    print("-" * 80)

    metrics_to_compare = [
        ('Final Value', 'final_value', '${:,.0f}'),
        ('Total Return', 'total_return', '{:.2f}%'),
        ('Annual Return', 'annual_return', '{:.2f}%'),
        ('Annual Volatility', 'annual_vol', '{:.2f}%'),
        ('Sharpe Ratio', 'sharpe', '{:.3f}'),
        ('Maximum Drawdown', 'max_dd', '{:.2f}%'),
        ('Calmar Ratio', 'calmar', '{:.3f}'),
        ('Win Rate', 'win_rate', '{:.2f}%')
    ]

    for label, key, fmt in metrics_to_compare:
        strat_val = strategy_metrics[key]
        bench_val = benchmark_metrics[key]

        # Calculate difference
        if key == 'final_value':
            diff = strat_val - bench_val
            diff_str = f"${diff:,.0f}"
        else:
            diff = strat_val - bench_val
            diff_str = f"{diff:+.2f}"

        print(f"{label:<25} {fmt.format(strat_val):>15} {fmt.format(bench_val):>15} {diff_str:>15}")

    print("-" * 80)


# ============================================================================
# VISUALIZATION WITH BENCHMARK
# ============================================================================

def plot_regime_portfolio_with_benchmark(portfolio_history, benchmark_df):
    """
    Visualize regime-conditional portfolio performance with benchmark comparison.

    Creates 5 comprehensive subplots:
    1. Portfolio Value: Strategy vs Benchmark equity curves (regime-colored)
    2. Regime Sequence: Timeline of detected regimes
    3. Cumulative Returns: Strategy vs Benchmark
    4. Rolling Sharpe Ratio: 63-day window (quarterly)
    5. Drawdown: Strategy vs Benchmark
    """

    portfolio_df = pd.DataFrame(portfolio_history)
    portfolio_df = portfolio_df[portfolio_df['regime'].notna()]

    fig, axes = plt.subplots(5, 1, figsize=(16, 18))

    regime_colors = {0: '#90EE90', 1: '#FFD700', 2: '#FF6B6B'}

    # ========================================================================
    # PLOT 1: PORTFOLIO VALUE - STRATEGY VS BENCHMARK
    # ========================================================================

    # Strategy equity curve with regime coloring
    for regime in [0, 1, 2]:
        mask = portfolio_df['regime'] == regime
        if mask.sum() > 0:
            axes[0].scatter(portfolio_df.loc[mask, 'date'],
                           portfolio_df.loc[mask, 'value'],
                           c=regime_colors[regime],
                           s=10,
                           alpha=0.6,
                           label=f"Strategy - {portfolio_df.loc[mask, 'regime_name'].iloc[0]}")

    # Benchmark equity curve
    axes[0].plot(benchmark_df['date'], benchmark_df['value'],
                color='black', linewidth=2, linestyle='--',
                label='Benchmark (URTH B&H)', alpha=0.7)

    axes[0].set_ylabel('Portfolio Value ($)', fontsize=11)
    axes[0].set_title('Strategy vs Benchmark: Equity Curves (Walk-Forward HMM Regime Detection)', 
                     fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper left', fontsize=9)
    axes[0].grid(alpha=0.3)

    # ========================================================================
    # PLOT 2: REGIME SEQUENCE
    # ========================================================================

    axes[1].scatter(portfolio_df['date'],
                   portfolio_df['regime'],
                   c=[regime_colors[r] for r in portfolio_df['regime']],
                   s=20,
                   alpha=0.6)
    axes[1].set_ylabel('Regime', fontsize=11)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['Low Vol\n(Max Sharpe)', 'Normal Vol\n(HRP)', 'High Vol\n(Risk Parity+SH)'])
    axes[1].set_title('Active Regime & Strategy (Walk-Forward HMM Detection)', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)

    # ========================================================================
    # PLOT 3: CUMULATIVE RETURNS
    # ========================================================================

    # Strategy cumulative returns
    portfolio_df['return'] = portfolio_df['value'].pct_change()
    strategy_cum_ret = portfolio_df['return'].cumsum() * 100

    # Benchmark cumulative returns
    benchmark_cum_ret = benchmark_df['return'].cumsum() * 100

    axes[2].plot(portfolio_df['date'], strategy_cum_ret, 
                color='blue', linewidth=2, label='Strategy', alpha=0.8)
    axes[2].plot(benchmark_df['date'], benchmark_cum_ret,
                color='black', linewidth=2, linestyle='--', label='Benchmark (URTH)', alpha=0.7)
    axes[2].set_ylabel('Cumulative Return (%)', fontsize=11)
    axes[2].set_title('Cumulative Returns: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    axes[2].axhline(y=0, color='black', linestyle='--', linewidth=1)

    # ========================================================================
    # PLOT 4: ROLLING SHARPE RATIO (63-day window = ~1 quarter)
    # ========================================================================

    # Strategy rolling Sharpe
    strategy_rolling_sharpe = (
        portfolio_df['return'].rolling(63).mean() * 252 / 
        (portfolio_df['return'].rolling(63).std() * np.sqrt(252))
    )

    # Benchmark rolling Sharpe
    benchmark_rolling_sharpe = (
        benchmark_df['return'].rolling(63).mean() * 252 / 
        (benchmark_df['return'].rolling(63).std() * np.sqrt(252))
    )

    axes[3].plot(portfolio_df['date'], strategy_rolling_sharpe,
                color='blue', linewidth=1.5, label='Strategy', alpha=0.8)
    axes[3].plot(benchmark_df['date'], benchmark_rolling_sharpe,
                color='black', linewidth=1.5, linestyle='--', label='Benchmark (URTH)', alpha=0.7)
    axes[3].set_ylabel('Rolling Sharpe (63d)', fontsize=11)
    axes[3].set_title('Rolling Sharpe Ratio: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[3].axhline(y=0, color='red', linestyle='--', linewidth=1)
    axes[3].legend()
    axes[3].grid(alpha=0.3)

    # ========================================================================
    # PLOT 5: DRAWDOWN COMPARISON
    # ========================================================================

    # Strategy drawdown
    strategy_returns = portfolio_df['return'].dropna()
    strategy_cumulative = (1 + strategy_returns).cumprod()
    strategy_running_max = strategy_cumulative.expanding().max()
    strategy_drawdown = (strategy_cumulative - strategy_running_max) / strategy_running_max

    # Benchmark drawdown
    benchmark_returns = benchmark_df['return'].dropna()
    benchmark_cumulative = (1 + benchmark_returns).cumprod()
    benchmark_running_max = benchmark_cumulative.expanding().max()
    benchmark_drawdown = (benchmark_cumulative - benchmark_running_max) / benchmark_running_max

    axes[4].fill_between(portfolio_df['date'].iloc[1:], strategy_drawdown.values * 100, 0,
                        color='blue', alpha=0.5, label='Strategy')
    axes[4].fill_between(benchmark_df['date'].iloc[1:], benchmark_drawdown.values * 100, 0,
                        color='black', alpha=0.3, label='Benchmark (URTH)')
    axes[4].set_ylabel('Drawdown (%)', fontsize=11)
    axes[4].set_xlabel('Date', fontsize=11)
    axes[4].set_title('Drawdown Comparison: Strategy vs Benchmark', fontsize=14, fontweight='bold')
    axes[4].legend()
    axes[4].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('regime_conditional_portfolio_hmm_risk_parity_short.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Visualization saved as 'regime_conditional_portfolio_hmm_risk_parity_short.png'")
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Execute regime-conditional portfolio system with walk-forward HMM analysis.

    Main execution flow:
    --------------------
    1. Fetch data (SPY for regimes, VIX for volatility, all assets, URTH for benchmark)
    2. Prepare HAR-semivariance features (with proper lagging)
    3. Run walk-forward backtest with HMM regime detection
    4. Create Buy & Hold benchmark
    5. Calculate and compare metrics
    6. Visualize results

    Strategy Summary:
    -----------------
    - Low Volatility (0): Max Sharpe on growth assets
    - Normal Volatility (1): HRP on growth + defensive (NO SH)
    - High Volatility (2): Risk Parity with short exposure (SH)
    """

    print("="*80)
    print("REGIME-CONDITIONAL PORTFOLIO WITH HMM (NO LOOK-AHEAD)")
    print("WITH URTH BUY & HOLD BENCHMARK COMPARISON")
    print("="*80)
    print("\nSTRATEGY MODIFICATION:")
    print("  Low Vol: Max Sharpe (growth assets)")
    print("  Normal Vol: HRP (growth + defensive, NO SH)")
    print("  High Vol: RISK PARITY + SHORT (defensive + SH)")
    print("="*80)

    try:
        # ====================================================================
        # DATA ACQUISITION
        # ====================================================================

        print(f"\nFetching {REGIME_TICKER} for regime detection...")
        spy_data = fetch_price_data(REGIME_TICKER, START_DATE, END_DATE)

        vix_data = None
        if USE_VIX:
            print("Fetching VIX...")
            try:
                vix_raw = fetch_price_data('%5EVIX', START_DATE, END_DATE)
                if vix_raw is not None:
                    vix_data = vix_raw['close']
            except:
                print("  Warning: Could not fetch VIX data, proceeding without it")

        print(f"\nFetching {BENCHMARK_TICKER} for benchmark...")
        benchmark_data = fetch_price_data(BENCHMARK_TICKER, START_DATE, END_DATE)
        if benchmark_data is None:
            print(f"  ✗ Failed to fetch {BENCHMARK_TICKER} - proceeding without benchmark")

        asset_data = fetch_all_assets(ALL_ASSETS, START_DATE, END_DATE)

        # ====================================================================
        # FEATURE PREPARATION
        # ====================================================================

        regime_features = prepare_regime_features(spy_data, vix_data)

        # ====================================================================
        # STRATEGY BACKTEST
        # ====================================================================

        backtest = RegimePortfolioBacktest_WalkForwardHMM(
            regime_features=regime_features,
            asset_data=asset_data,
            initial_capital=INITIAL_CAPITAL,
            transaction_cost=TRANSACTION_COST
        )

        portfolio_history, trades = backtest.run_backtest()

        # ====================================================================
        # BENCHMARK CREATION
        # ====================================================================

        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df = portfolio_df[portfolio_df['regime'].notna()]

        benchmark_df = None
        if benchmark_data is not None:
            benchmark_df = create_buy_hold_benchmark(
                benchmark_data=benchmark_data,
                regime_dates=portfolio_df['date'].tolist(),
                initial_capital=INITIAL_CAPITAL,
                transaction_cost=TRANSACTION_COST
            )

        # ====================================================================
        # PERFORMANCE METRICS
        # ====================================================================

        print(f"\n{'='*80}")
        print("STRATEGY PERFORMANCE SUMMARY")
        print(f"{'='*80}")

        # Calculate strategy metrics
        strategy_metrics = calculate_performance_metrics(portfolio_df, INITIAL_CAPITAL)

        print(f"\nStrategy Performance:")
        print(f"  Initial Capital:   ${INITIAL_CAPITAL:,.0f}")
        print(f"  Final Value:       ${strategy_metrics['final_value']:,.0f}")
        print(f"  Total Return:      {strategy_metrics['total_return']:.2f}%")
        print(f"  Annual Return:     {strategy_metrics['annual_return']:.2f}%")
        print(f"  Annual Volatility: {strategy_metrics['annual_vol']:.2f}%")
        print(f"  Sharpe Ratio:      {strategy_metrics['sharpe']:.3f}")
        print(f"  Maximum Drawdown:  {strategy_metrics['max_dd']:.2f}%")
        print(f"  Calmar Ratio:      {strategy_metrics['calmar']:.3f}")
        print(f"  Win Rate:          {strategy_metrics['win_rate']:.2f}%")
        print(f"  Total Trades:      {len(trades)}")

        # Calculate and compare benchmark metrics
        if benchmark_df is not None:
            benchmark_metrics = calculate_performance_metrics(benchmark_df, INITIAL_CAPITAL)
            print_comparative_metrics(strategy_metrics, benchmark_metrics)

        # ====================================================================
        # REGIME STATISTICS
        # ====================================================================

        print(f"\n{'='*80}")
        print("REGIME STATISTICS (Walk-Forward HMM Detection)")
        print(f"{'='*80}")

        for regime in [0, 1, 2]:
            regime_mask = portfolio_df['regime'] == regime
            if regime_mask.sum() > 0:
                regime_name = portfolio_df.loc[regime_mask, 'regime_name'].iloc[0]
                n_days = regime_mask.sum()
                pct = n_days / len(portfolio_df) * 100

                # Calculate regime-specific performance
                regime_returns = portfolio_df.loc[regime_mask, 'value'].pct_change().dropna()
                regime_sharpe = (regime_returns.mean() * 252) / (regime_returns.std() * np.sqrt(252)) if len(regime_returns) > 0 else 0

                print(f"\nRegime {regime} ({regime_name}):  ")
                print(f"  Days:         {n_days} ({pct:.1f}%)")
                print(f"  Sharpe Ratio: {regime_sharpe:.3f}")

        # ====================================================================
        # VISUALIZATION
        # ====================================================================

        if benchmark_df is not None:
            plot_regime_portfolio_with_benchmark(portfolio_history, benchmark_df)
        else:
            print("\n✗ Skipping benchmark visualization (no benchmark data)")

        print(f"\n{'='*80}")
        print("✓ EXECUTION COMPLETE (NO LOOK-AHEAD BIAS)")
        print(f"{'='*80}")

        return portfolio_history, trades, backtest.regime_history

    except Exception as e:
        print(f"\nERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None


if __name__ == "__main__":
    portfolio_history, trades, regime_history = main()
